# Hispasonic — Supply & Demand Analysis

**Source:** `../data/processed/hispasonic_unified.csv`  
**Goal:** Determine the relationship between supply, demand, city and price in the second-hand synthesizer market  
**Figures:** Saved automatically to `../reports/figures/`

---

## Definitions

| Concept | Column(s) | Logic |
|---------|-----------|-------|
| **Supply** | `sell == 1` | Someone offers an item for sale |
| **Demand** | `buy == 1` or `search == 1` | Someone is actively looking to buy |
| **S/D ratio** | demand / supply | > 1 → more demand than supply (seller's market) |
| **Proxy demand** | `seen` | Total views — passive interest regardless of listing type |

**Hypothesis:** Cities with a high demand/supply ratio should show higher median sell prices.  
**Temporal hypothesis:** When the ratio rises in a city across scrape dates, prices should follow upward.

---

## Analysis plan
1. Imports and config
2. Load dataset
3. Supply and demand counts per city
4. Supply/demand ratio per city — ranking and price correlation
5. Temporal evolution of supply, demand and ratio per city
6. Price vs ratio scatter — does ratio predict price?
7. Supply/demand ratio over time — market tightness evolution
8. `seen` as passive demand proxy — city and brand breakdown
9. Summary findings

## 1. Imports and config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
from scipy import stats

DATA_PATH   = '../data/processed/hispasonic_unified.csv'
FIGURES_DIR = '../reports/figures/'
os.makedirs(FIGURES_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

TOP_N = 15
MIN_SUPPLY = 5  # minimum supply listings to include a city in ratio analysis

def savefig(name):
    path = os.path.join(FIGURES_DIR, f'{name}.png')
    plt.savefig(path, dpi=150)
    print(f'Saved → {path}')
    plt.show()

print('Setup complete.')

## 2. Load dataset

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['published', 'expire', 'date_scrapped'])

# Derived columns
df['is_supply'] = df['sell'].fillna(0).astype(int)
df['is_demand'] = ((df['buy'].fillna(0) + df['search'].fillna(0)) > 0).astype(int)

print(f'Shape: {df.shape}')
print(f'Supply listings: {df["is_supply"].sum():,}')
print(f'Demand listings: {df["is_demand"].sum():,}')
print(f'Scrape dates: {df["date_scrapped"].nunique()}')
print(f'Cities: {df["city"].nunique()}')

## 3. Supply and demand counts per city

In [ ]:
# Aggregate by city
city_agg = df.groupby('city').agg(
    supply   = ('is_supply', 'sum'),
    demand   = ('is_demand', 'sum'),
    total    = ('is_supply', 'count'),
    med_price= ('price', 'median'),
    mean_seen= ('seen', 'mean')
).reset_index()

# Filter cities with enough supply to be meaningful
city_agg = city_agg[city_agg['supply'] >= MIN_SUPPLY].copy()

# Supply/demand ratio (demand per unit of supply)
city_agg['sd_ratio'] = city_agg['demand'] / city_agg['supply'].replace(0, np.nan)

city_agg = city_agg.sort_values('total', ascending=False)

print(f'Cities with ≥{MIN_SUPPLY} supply listings: {len(city_agg)}')
city_agg.head(15)

In [ ]:
# ── 3a. Supply vs demand side by side — top cities ───────────────────────
top_cities = city_agg.nlargest(TOP_N, 'total')

x = np.arange(len(top_cities))
w = 0.38

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - w/2, top_cities['supply'], width=w, label='Supply (sell)',
       color=sns.color_palette('muted')[0])
ax.bar(x + w/2, top_cities['demand'], width=w, label='Demand (buy + search)',
       color=sns.color_palette('muted')[1])
ax.set_xticks(x)
ax.set_xticklabels(top_cities['city'], rotation=45, ha='right')
ax.set_title(f'Supply vs Demand — top {TOP_N} cities by total listings', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of listings')
ax.legend()
savefig('19_supply_vs_demand_by_city')

## 4. Supply/demand ratio per city — ranking and price correlation

In [ ]:
# ── 4a. S/D ratio ranking ────────────────────────────────────────────────
ratio_ranked = city_agg.dropna(subset=['sd_ratio']).nlargest(TOP_N, 'sd_ratio')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if r > 1 else '#3498db' for r in ratio_ranked['sd_ratio']]
bars = ax.barh(ratio_ranked['city'], ratio_ranked['sd_ratio'], color=colors)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.2, label='Balanced (ratio = 1)')
ax.set_title(f'Supply/demand ratio — top {TOP_N} cities\n'
             'Red = seller\'s market (demand > supply) | Blue = buyer\'s market',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Demand / Supply ratio')
ax.set_ylabel('')
ax.legend()
savefig('20_sd_ratio_ranking_by_city')

In [ ]:
# ── 4b. S/D ratio vs median price — scatter with regression ──────────────
plot_df = city_agg.dropna(subset=['sd_ratio', 'med_price'])
plot_df = plot_df[plot_df['med_price'] > 0]

# Pearson correlation
r, p = stats.pearsonr(plot_df['sd_ratio'], plot_df['med_price'])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(plot_df['sd_ratio'], plot_df['med_price'],
           s=plot_df['total'] * 0.8, alpha=0.6,
           color=sns.color_palette('muted')[2], edgecolors='white', linewidth=0.5)

# Regression line
m, b = np.polyfit(plot_df['sd_ratio'], plot_df['med_price'], 1)
x_line = np.linspace(plot_df['sd_ratio'].min(), plot_df['sd_ratio'].max(), 100)
ax.plot(x_line, m * x_line + b, color='tomato', linewidth=2, linestyle='--')

# Label top cities
for _, row in plot_df.nlargest(6, 'total').iterrows():
    ax.annotate(row['city'], (row['sd_ratio'], row['med_price']),
                fontsize=8, ha='left', va='bottom',
                xytext=(4, 4), textcoords='offset points')

ax.axvline(1.0, color='grey', linestyle=':', linewidth=1)
ax.set_title(f'S/D ratio vs median price per city\n'
             f'Pearson r = {r:.3f}  |  p = {p:.4f}  |  bubble size = total listings',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Demand / Supply ratio')
ax.set_ylabel('Median sell price (€)')
savefig('21_sd_ratio_vs_price_scatter')

In [ ]:
# ── 4c. Dual axis: ratio and price by city ────────────────────────────────
dual_df = city_agg.dropna(subset=['sd_ratio', 'med_price']).nlargest(12, 'total').sort_values('sd_ratio', ascending=False)

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

x = np.arange(len(dual_df))
ax1.bar(x, dual_df['sd_ratio'], color=sns.color_palette('muted')[0], alpha=0.7, label='S/D ratio')
ax2.plot(x, dual_df['med_price'], color='tomato', marker='o', linewidth=2, label='Median price (€)')

ax1.axhline(1.0, color='black', linestyle='--', linewidth=1)
ax1.set_xticks(x)
ax1.set_xticklabels(dual_df['city'], rotation=45, ha='right')
ax1.set_ylabel('Demand / Supply ratio', color=sns.color_palette('muted')[0])
ax2.set_ylabel('Median sell price (€)', color='tomato')
ax1.set_title('Supply/demand ratio and median price — top 12 cities', fontsize=14, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
savefig('22_sd_ratio_and_price_dual_axis')

## 5. Temporal evolution of supply, demand and ratio per city

In [ ]:
# Aggregate by city + scrape date
time_city = df.groupby(['city', 'date_scrapped']).agg(
    supply    = ('is_supply', 'sum'),
    demand    = ('is_demand', 'sum'),
    med_price = ('price', 'median')
).reset_index()

time_city['sd_ratio'] = time_city['demand'] / time_city['supply'].replace(0, np.nan)

# Pick top 5 cities by total listings for temporal charts
top5_cities = city_agg.nlargest(5, 'total')['city'].tolist()
print('Top 5 cities for temporal analysis:', top5_cities)

In [ ]:
# ── 5a. Supply and demand over time per city ──────────────────────────────
fig, axes = plt.subplots(len(top5_cities), 1, figsize=(12, 3.5 * len(top5_cities)), sharex=True)

for ax, city in zip(axes, top5_cities):
    sub = time_city[time_city['city'] == city].sort_values('date_scrapped')
    ax.plot(sub['date_scrapped'], sub['supply'], marker='o', label='Supply',
            color=sns.color_palette('muted')[0], linewidth=1.8)
    ax.plot(sub['date_scrapped'], sub['demand'], marker='s', label='Demand',
            color=sns.color_palette('muted')[1], linewidth=1.8, linestyle='--')
    ax.set_title(city, fontweight='bold')
    ax.set_ylabel('Listings')
    ax.legend(loc='upper right', fontsize=9)
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))

plt.xticks(rotation=45)
fig.suptitle('Supply vs Demand over time — top 5 cities', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('23_supply_demand_over_time_by_city')

In [ ]:
# ── 5b. S/D ratio evolution over time per city ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
palette = sns.color_palette('tab10', n_colors=len(top5_cities))

for i, city in enumerate(top5_cities):
    sub = time_city[time_city['city'] == city].sort_values('date_scrapped').dropna(subset=['sd_ratio'])
    ax.plot(sub['date_scrapped'], sub['sd_ratio'], marker='o',
            label=city, color=palette[i], linewidth=1.8)

ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Balance (ratio = 1)')
ax.set_title('Supply/demand ratio over time — top 5 cities', fontsize=14, fontweight='bold')
ax.set_xlabel('Scrape date')
ax.set_ylabel('Demand / Supply ratio')
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
savefig('24_sd_ratio_evolution_by_city')

In [ ]:
# ── 5c. Does the ratio predict next-period price? (lagged correlation) ────
# For each city, correlate sd_ratio(t) with med_price(t+1)
lag_results = []

for city in city_agg['city']:
    sub = time_city[time_city['city'] == city].sort_values('date_scrapped').dropna(subset=['sd_ratio', 'med_price'])
    if len(sub) < 4:
        continue
    ratio_t  = sub['sd_ratio'].values[:-1]
    price_t1 = sub['med_price'].values[1:]
    if len(ratio_t) < 3:
        continue
    r, p = stats.pearsonr(ratio_t, price_t1)
    lag_results.append({'city': city, 'lag_r': r, 'lag_p': p, 'n': len(ratio_t)})

lag_df = pd.DataFrame(lag_results).sort_values('lag_r', ascending=False)
print('Cities where S/D ratio (t) correlates with price (t+1):')
print(lag_df.to_string(index=False))

## 6. S/D ratio over time — global market tightness

In [ ]:
# ── 6a. Global supply, demand and ratio over time ─────────────────────────
global_time = df.groupby('date_scrapped').agg(
    supply    = ('is_supply', 'sum'),
    demand    = ('is_demand', 'sum'),
    med_price = ('price', 'median')
).reset_index()

global_time['sd_ratio'] = global_time['demand'] / global_time['supply'].replace(0, np.nan)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top: supply and demand volumes
ax1.plot(global_time['date_scrapped'], global_time['supply'], marker='o',
         label='Supply', color=sns.color_palette('muted')[0], linewidth=2)
ax1.plot(global_time['date_scrapped'], global_time['demand'], marker='s',
         label='Demand', color=sns.color_palette('muted')[1], linewidth=2, linestyle='--')
ax1.set_ylabel('Number of listings')
ax1.set_title('Global market: supply, demand and S/D ratio over time',
              fontsize=14, fontweight='bold')
ax1.legend()

# Bottom: ratio + price
ax3 = ax2.twinx()
ax2.bar(global_time['date_scrapped'], global_time['sd_ratio'],
        width=10, color=sns.color_palette('muted')[2], alpha=0.6, label='S/D ratio')
ax3.plot(global_time['date_scrapped'], global_time['med_price'],
         color='tomato', marker='o', linewidth=2, label='Median price (€)')
ax2.axhline(1.0, color='black', linestyle='--', linewidth=1)
ax2.set_ylabel('Demand / Supply ratio', color=sns.color_palette('muted')[2])
ax3.set_ylabel('Median price (€)', color='tomato')
ax2.set_xlabel('Scrape date')
ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)

lines2, labels2 = ax2.get_legend_handles_labels()
lines3, labels3 = ax3.get_legend_handles_labels()
ax2.legend(lines2 + lines3, labels2 + labels3, loc='upper right')

plt.tight_layout()
savefig('25_global_supply_demand_ratio_price')

## 7. `seen` as passive demand proxy

In [ ]:
# ── 7a. Seen vs price by city ─────────────────────────────────────────────
seen_city = df.groupby('city').agg(
    mean_seen = ('seen', 'mean'),
    med_price = ('price', 'median'),
    total     = ('seen', 'count')
).reset_index().dropna()
seen_city = seen_city[seen_city['total'] >= MIN_SUPPLY]

r2, p2 = stats.pearsonr(seen_city['mean_seen'], seen_city['med_price'])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(seen_city['mean_seen'], seen_city['med_price'],
           s=seen_city['total'] * 0.5, alpha=0.6,
           color=sns.color_palette('muted')[4], edgecolors='white')

m2, b2 = np.polyfit(seen_city['mean_seen'], seen_city['med_price'], 1)
x2 = np.linspace(seen_city['mean_seen'].min(), seen_city['mean_seen'].max(), 100)
ax.plot(x2, m2 * x2 + b2, color='tomato', linewidth=2, linestyle='--')

for _, row in seen_city.nlargest(6, 'total').iterrows():
    ax.annotate(row['city'], (row['mean_seen'], row['med_price']),
                fontsize=8, ha='left', va='bottom',
                xytext=(4, 4), textcoords='offset points')

ax.set_title(f'Mean views (seen) vs median price per city\n'
             f'Pearson r = {r2:.3f}  |  p = {p2:.4f}  |  bubble size = listings',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean views per listing')
ax.set_ylabel('Median price (€)')
savefig('26_seen_vs_price_by_city')

## 8. Summary findings

In [ ]:
print('=' * 60)
print('SUPPLY / DEMAND ANALYSIS — KEY FINDINGS')
print('=' * 60)

print(f'\nTotal supply listings : {df["is_supply"].sum():,}')
print(f'Total demand listings : {df["is_demand"].sum():,}')
print(f'Global S/D ratio      : {df["is_demand"].sum() / df["is_supply"].sum():.3f}')

print('\n── Top 5 seller markets (highest demand/supply ratio) ──')
print(city_agg.nlargest(5, 'sd_ratio')[['city','supply','demand','sd_ratio','med_price']].to_string(index=False))

print('\n── Top 5 buyer markets (lowest demand/supply ratio) ───')
print(city_agg.nsmallest(5, 'sd_ratio')[['city','supply','demand','sd_ratio','med_price']].to_string(index=False))

print(f'\n── S/D ratio vs median price correlation ──────────────')
print(f'Pearson r = {r:.3f}  |  p-value = {p:.4f}')
if p < 0.05:
    direction = 'positive' if r > 0 else 'negative'
    print(f'→ Statistically significant {direction} correlation.')
    if r > 0:
        print('→ Cities with more demand relative to supply tend to have higher prices.')
    else:
        print('→ Cities with more demand relative to supply tend to have lower prices.')
else:
    print('→ No statistically significant linear relationship found.')

figs = [f for f in sorted(os.listdir(FIGURES_DIR)) if f.startswith('1') or f[0] in '23']
new_figs = [f for f in sorted(os.listdir(FIGURES_DIR)) if f[:2] in ['19','20','21','22','23','24','25','26']]
print(f'\n── Figures saved ──────────────────────────────────────')
for f in new_figs:
    print(f'  {f}')